<a href="https://colab.research.google.com/github/demekeendalie/multi-emotion-classification/blob/main/XLM_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Try reading the CSV with different encodings
data = pd.read_excel('/content/drive/MyDrive/preprocessed emotion_dataset.xlsx')
display(data.tail(10))

,comments,sad,happy,anger,fear,disgust,surprise,contempt,neutral
22020,አሁንም ዐቢይ አህመድ ኢትዮጵያን ቅስሟን ሰብሬ እጥላለው እያለ እየታገለ ...,0,0,1,1,1,0,0,0
22021,98 የወለጋ የተፈናቀለው የተገደለው ሙስሊም አማራ ነው ኦሮሞ እና ሙስሊም...,0,0,1,0,1,1,0,0
22022,ጠላት ደነገጠ በባህር ኃይላችን ፐ ፐ ፐ ባህር ኃይል ችግኝ ቤተመንግስት ...,1,0,0,0,0,1,0,0
22023,በዘር ጥላቻ ተወጥረህ መቸም አይገባህም ስለአገር ልክልክህን ነግረውሃ ረጅ...,0,0,1,0,1,0,0,0
22024,እርፍ በል ሽሜ ሊያውም ሙሉ ብሔራዊ ቡድኑን ነው ያደመከው። የሚገርም ህብ...,0,0,1,0,0,0,0,0
22025,ለኔ የምንግዜም ጀግና ሁሌም አምንህ ነበር ሁሉም በአብይ ፍቅር ባበደበት ...,1,0,0,0,0,1,0,0
22026,የአብይ ጥበቃ አራዊት ከብልፅግና በላይ ፅንፈኛ ከብልፅግና በላይ የኢትዮጵ...,0,0,1,0,1,0,0,0
22027,አለምነህ ዋሴ ምርጥ ይሁዳዊ ጋዜጠኛ በርታልን እኔ የኛ ሰው መሆንክን ባወ...,1,0,0,0,0,1,0,0
22028,ሰው ሁሉ እንደ እርስዎ አላዋቂ፣ ግብዝ እና ተግባር በሌለው ተንኮልን ባነ...,0,0,1,0,1,0,0,0
22029,ሶፎንያስ በሕይወት ሲኖር ያማልዳል !!!ከሞት ቦኃላ ወደ እንጦርጦስ ስለሆ...,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Extract label columns from the DataFrame
label_columns = ['sad', 'happy', 'anger', 'fear', 'disgust', 'surprise', 'contempt', 'neutral']
labels = data[label_columns]

# The labels are already in binary format, so we can directly use their values.
binary_labels = labels.values

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

# First split: data and corresponding binary_labels
train_val_df, test_dataset, train_val_labels, test_labels = train_test_split(
    data, binary_labels, test_size=0.10, random_state=42
)

# Second split: train_val_df and train_val_labels
train_dataset, evaluation_dataset, train_labels, val_labels = train_test_split(
    train_val_df, train_val_labels, test_size=0.10, random_state=42
)

print('Training dataset shape: ', train_dataset.shape)
print('Validation dataset shape: ', evaluation_dataset.shape)
print('Testing dataset shape: ', test_dataset.shape)

Training dataset shape:  (17844, 9)
Validation dataset shape:  (1983, 9)
Testing dataset shape:  (2203, 9)


In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Extract texts from the datasets
train_texts = train_dataset['comments'].tolist()
val_texts = evaluation_dataset['comments'].tolist()
test_texts = test_dataset['comments'].tolist()

# Tokenize the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings=tokenizer(test_texts, truncation=True, padding=True, max_length=512)

In [ ]:
# Create PyTorch Dataset
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)  # Use float for multi-label
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MultiLabelDataset(train_encodings, train_labels)
val_dataset = MultiLabelDataset(val_encodings, val_labels)
test_dataset = MultiLabelDataset(test_encodings, test_labels)

In [ ]:
# Load model
model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_columns),
    problem_type="multi_label_classification"  # Set problem type
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Set training arguments
training_args = TrainingArguments(
   output_dir="./multi_emotion",
   eval_strategy='epoch',
   save_strategy='epoch',
   logging_strategy='epoch',
   num_train_epochs=5,
   learning_rate=1e-6,
   per_device_train_batch_size=4,  # batch size per device during training
   per_device_eval_batch_size=4,   # batch size for evaluation
   warmup_steps=1000,                # number of warmup steps for learning rate
   weight_decay=0.01,
   logging_dir='./logs',            # directory for storing logs (Corrected parameter name)
   logging_steps=20,
   report_to="none",
   load_best_model_at_end= True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss
)
def custom_metrics(eval_pred):
    logits, labels = eval_pred

    # Apply sigmoid to logits to get probabilities for each label
    probabilities = torch.sigmoid(torch.from_numpy(logits)).numpy()
    # Convert probabilities to binary predictions using a threshold (e.g., 0.5)
    predictions = (probabilities >= 0.5).astype(int)

    # Ensure labels are integers (they were floats from binary_labels.values)
    references = labels.astype(int)

    # Compute metrics using sklearn.metrics, which explicitly supports multi-label
    f1 = f1_score(references, predictions, average="micro")
    precision = precision_score(references, predictions, average="micro")
    recall = recall_score(references, predictions, average="micro")
    accuracy = accuracy_score(references, predictions)
    hamming = hamming_loss(references, predictions)

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "hamming_loss": hamming
    }

In [ ]:
# Trainer
from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=custom_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.548315,0.373183,0.785228,0.939098,0.674683,0.284922,0.139309,7.962900,249.029000,62.289000
2,0.356362,0.329947,0.802870,0.957225,0.691383,0.307615,0.128152,7.885800,251.465000,62.898000
3,0.319052,0.299990,0.828685,0.930697,0.746827,0.374181,0.116553,7.915900,250.508000,62.659000
4,0.303183,0.288554,0.834636,0.928236,0.758183,0.393848,0.113401,7.941100,249.715000,62.460000
5,0.297912,0.284233,0.839040,0.928658,0.765197,0.401916,0.110817,7.930200,250.057000,62.546000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=22305, training_loss=0.36496507054191885, metrics={'train_runtime': 1296.5498, 'train_samples_per_second': 68.813, 'train_steps_per_second': 17.203, 'total_flos': 1.173801649102848e+16, 'train_loss': 0.36496507054191885, 'epoch': 5.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

{'eval_loss': 0.2927159070968628, 'eval_f1': 0.8298497439771678, 'eval_precision': 0.925482119453286, 'eval_recall': 0.7521302495435179, 'eval_accuracy': 0.3908306854289605, 'eval_hamming_loss': 0.11501361779391739, 'eval_runtime': 8.9073, 'eval_samples_per_second': 247.325, 'eval_steps_per_second': 61.859, 'epoch': 5.0}


In [ ]:
print("Evaluation metrics:")
for key, value in test_results.items():
    print(f"{key}: {value}")

Evaluation metrics:
eval_loss: 0.2927159070968628
eval_f1: 0.8298497439771678
eval_precision: 0.925482119453286
eval_recall: 0.7521302495435179
eval_accuracy: 0.3908306854289605
eval_hamming_loss: 0.11501361779391739
eval_runtime: 8.9073
eval_samples_per_second: 247.325
eval_steps_per_second: 61.859
epoch: 5.0
